In [1]:
import cv2
import torch
import numpy as np
from pathlib import Path
from PIL import Image
import os
import tempfile

from anomalib.models import Padim
from anomalib.engine import Engine
from anomalib.data import Folder
from anomalib.data import PredictDataset

from lightning.pytorch.callbacks import ModelCheckpoint


W0804 20:22:17.869000 21796 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
d:\Projects\VisionXM\.visionXM\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Projects\VisionXM\.visionXM\lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = CHECKPOINT_DIR / "padim_checkpoint.ckpt"

print(DATA_DIR)
print(DATA_DIR.exists())


d:\Projects\VisionXM\data
True


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)


Using: cuda


In [4]:
datamodule = Folder(
    name="screws",
    root=str(DATA_DIR),
    normal_dir="train/good",
    abnormal_dir=[
        "test/manipulated_front",
        "test/scratch_head",
        "test/scratch_neck",
        "test/thread_side",
        "test/thread_top",
    ],
    normal_test_dir="test/good",
    train_batch_size=32,
    eval_batch_size=32,
    num_workers=4,
)


In [5]:
model = Padim(backbone="resnet18", layers=["layer1", "layer2", "layer3"])


In [6]:
# FIX: accelerator is now chosen dynamically instead of being hardcoded to "gpu",
# which crashed on any machine without a CUDA GPU.
engine = Engine(accelerator="gpu" if torch.cuda.is_available() else "cpu")

engine.fit(model=model, datamodule=datamodule)

# FIX: use pathlib instead of a non-raw Windows path string (".\checkpoints\..."),
# which is fragile and not cross-platform. Directory is created above.
engine.trainer.save_checkpoint(str(CHECKPOINT_PATH))
print("Checkpoint saved to:", CHECKPOINT_PATH)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..
You are using a CUDA device ('NVIDIA GeForce RTX 5070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
d:\Projects\VisionXM\.visionXM\lib\site-packages\lightning\pytorch\core\optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run wi

┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor  │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator     │      0 │ train │     0 │
│ 3 │ model          │ PadimModel    │  2.8 M │ train │     0 │
└───┴────────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 2.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.8 M                                                                                                
Total estimated model params size (MB): 11.131                                                                     
Modules in train mode: 19                                                                                          
Modules in eval mode: 69                                                                                           
Total FLOPs: 0

d:\Projects\VisionXM\.visionXM\lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
d:\Projects\VisionXM\.visionXM\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


d:\Projects\VisionXM\.visionXM\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')

d:\Projects\VisionXM\.visionXM\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.
d:\Projects\VisionXM\.visionXM\lib\site-packages\lightning\pytorch\loops\fit_loop.py:538: Found 69 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.
`Trainer.fit` stopped: `max_epochs=1` reached.


`weights_only` was not set, defaulting to `False`.


Checkpoint saved to: d:\Projects\VisionXM\checkpoints\padim_checkpoint.ckpt


In [7]:
engine.test(model=model, datamodule=datamodule)


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
d:\Projects\VisionXM\.visionXM\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'test_dataloader' to speed up the dataloader worker initialization.


d:\Projects\VisionXM\.visionXM\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: The ``compute`` 
method of metric AUROC was called before the ``update`` method which may lead to errors, as metric states have not 
yet been updated.
  warnings.warn(*args, **kwargs)

d:\Projects\VisionXM\.visionXM\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: The ``compute`` 
method of metric F1Score was called before the ``update`` method which may lead to errors, as metric states have 
not yet been updated.
  warnings.warn(*args, **kwargs)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │    0.7904762029647827     │
│       image_F1Score       │     0.885496199131012     │
└───────────────────────────┴───────────────────────────┘

[{'image_AUROC': 0.7904762029647827, 'image_F1Score': 0.885496199131012}]

In [8]:
predictions = engine.predict(
    model=model,
    datamodule=datamodule,
    ckpt_path=str(CHECKPOINT_PATH),
)


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
Restoring states from the checkpoint path at D:\Projects\VisionXM\checkpoints\padim_checkpoint.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at D:\Projects\VisionXM\checkpoints\padim_checkpoint.ckpt
d:\Projects\VisionXM\.visionXM\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'predict_dataloader' to speed up the dataloader worker initialization.


In [9]:
def predict_crop(crop):
    """
    Run PaDiM inference on a single cropped screw image and
    return (label, anomaly_score, box_color).
    """
    # FIX: previously this wrote into a real dataset file
    # (D:\Projects\VisionXM\data\test\thread_side\006.png) and then deleted it,
    # silently destroying a dataset image every time this ran.
    # FIX: the original path was also a *non-raw* string, so "\thread_side\006.png"
    # was corrupted by Python's escape processing (\t -> tab, \006 -> octal escape),
    # which is exactly what caused the
    # "ValueError: Path contains non-printable characters" seen below.
    # Using a real temp file avoids both problems.
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
        temp_file = tmp.name

    cv2.imwrite(temp_file, crop)

    dataset = PredictDataset(path=temp_file)

    predictions = engine.predict(
        model=model,
        dataset=dataset,
        ckpt_path=str(CHECKPOINT_PATH),
    )

    result = predictions[0]

    score = float(result.pred_score)

    label = "Defective" if result.pred_label else "Good"

    color = (0, 0, 255) if label == "Defective" else (0, 255, 0)

    os.remove(temp_file)

    return label, score, color


In [10]:
# FIX: hardcoded absolute Windows path replaced with a path built from DATA_DIR
# so this notebook works on any machine/OS as long as the dataset is in ../data
sample_path = DATA_DIR / "train" / "good" / "007.png"
img = cv2.imread(str(sample_path))

assert img is not None, f"Could not read image: {sample_path}"

label, score, color = predict_crop(img)

print(label)
print(score)


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
Restoring states from the checkpoint path at D:\Projects\VisionXM\checkpoints\padim_checkpoint.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at D:\Projects\VisionXM\checkpoints\padim_checkpoint.ckpt
d:\Projects\VisionXM\.visionXM\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Good
0.24097153544425964
